# Issue #10: Exploratory Data Analysis – Charging Demand Patterns

**Input:** `acn_clean_sessions.csv` and `acn_flagged_sessions.csv` (Issue #9, repository root)  
**Outputs (repository root):** `eda_tables/*.csv` (statistical summaries), `eda_key_findings.md` (numbers computed from the data)

This notebook is the *statistical* analysis: tables, tests and anomaly checks. The chart set is Issue #11.

**Reading rules used throughout**
- With tens of thousands of sessions almost any difference is "statistically significant". Tests are reported with an **effect size**; judge importance from the effect size and the medians, not from p-values.
- **Regimes.** The operator changed the rules twice (Lee, Li & Low, *ACN-Data: Analysis and Applications of an Open EV Charging Dataset*, e-Energy 2019, Sec. 2.2, 3.1.2 and Fig. 2): sessions without app input ("unclaimed", no `userID`) received a charger-specific default energy allowance until 31 Aug 2018 and 14 kWh over 12 h from 1 Sep 2018; from 1 Nov 2018 charging cost $0.12/kWh for claimed sessions and unclaimed sessions were stopped after 30 minutes. From mid-March 2020 the campus moved to remote operation (COVID-19). This gives four regimes: **R1** free, **R2** free with the 14 kWh default, **R3** paid, **R4** COVID period. The COVID start date is an assumption checked against the data in Section 0.
- **Scope.** Sections 0–7c describe **normal operation** (R1–R3, before the COVID start): pooling near-empty COVID days with normal days would drag every average toward zero. Section 7e compares all regimes and Section 7f describes the COVID period and recovery.
- **Data gaps.** Runs of 7 or more consecutive days with zero sessions are recording gaps, not zero demand. They are listed in Section 0b and excluded from all daily statistics.
- **Stay split.** Short vs. long stays use the density valley between the two connection-time modes, as in Issue #9.

In [1]:
import os
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

try:
    from scipy import stats
    HAS_SCIPY = True
except ImportError:
    HAS_SCIPY = False
    print("scipy not installed: statistical tests will be skipped (pip install scipy)")

pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
from pathlib import Path
def find_root(marker):
    """Repository root = the folder holding `marker` (this folder or its parent)."""
    for p in [Path.cwd(), Path.cwd().parent]:
        if (p / marker).exists():
            return p
    raise FileNotFoundError(f"{marker} not found in this folder or its parent")

ROOT = find_root("acn_clean_sessions.csv")
TAB_DIR = ROOT / "eda_tables"
TAB_DIR.mkdir(exist_ok=True)

def tab(name, frame, show=True):
    """Save a table to eda_tables/ and display it."""
    frame.to_csv(TAB_DIR / f"{name}.csv")
    if show:
        display(frame)
    return frame

LOCAL_TZ = "America/Los_Angeles"
DOW = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

df = pd.read_csv(ROOT / "acn_clean_sessions.csv")
for c in ["connectionTime", "disconnectTime", "doneChargingTime"]:
    df[c] = pd.to_datetime(df[c], utc=True)
    df[c + "_loc"] = df[c].dt.tz_convert(LOCAL_TZ).dt.tz_localize(None)

df["date"] = df["connectionTime_loc"].dt.normalize()
df["month"] = df["connectionTime_loc"].dt.to_period("M")
df["hour"] = df["connectionTime_loc"].dt.hour
df["dayofweek"] = df["connectionTime_loc"].dt.dayofweek
df["is_weekend"] = df["dayofweek"] >= 5
df["day_type"] = np.where(df["is_weekend"], "Weekend", "Weekday")
if "power_implausible" not in df.columns:
    df["power_implausible"] = False
df["done_ok"] = df["charging_h"].notna() & ~df["power_implausible"].astype(bool)
df["idle_h"] = np.where(df["done_ok"], (df["connected_h"] - df["charging_h"]).clip(lower=0), np.nan)
df["idle_share"] = df["idle_h"] / df["connected_h"]
# Short/long split at the density valley between the two connection-time modes (same method as Issue #9)
_h = df["connected_h"]
_counts, _edges = np.histogram(_h[(_h > 0) & (_h < 12)], bins=np.arange(0, 12.25, 0.25))
_smooth = pd.Series(_counts).rolling(5, center=True, min_periods=1).mean().values
_centres = (_edges[:-1] + _edges[1:]) / 2
_win = (_centres >= 1) & (_centres <= 10)
valley_h = float(_centres[_win][np.argmin(_smooth[_win])])
SHORT_STAY_H = round(valley_h * 2) / 2
SHORT_LABEL, LONG_LABEL = f"Short (<{SHORT_STAY_H:g} h)", f"Long (>={SHORT_STAY_H:g} h)"
df["stay"] = np.where(df["connected_h"] < SHORT_STAY_H, SHORT_LABEL, LONG_LABEL)

# Regimes. Policy dates: Lee, Li & Low 2019 (ACN-Data paper, Fig. 2, Sec. 3.1.2).
# COVID_START is an ASSUMPTION (campus moved to remote operation in mid-March 2020); it is checked against the data below.
R2_START, PAID_START, COVID_START = pd.Timestamp("2018-09-01"), pd.Timestamp("2018-11-01"), pd.Timestamp("2020-03-16")
POLICY_DATES = [R2_START, PAID_START, COVID_START]
REGIMES = ["R1 free", "R2 free, 14 kWh default", "R3 paid", "R4 COVID period"]
REGIME_DESC = {"R1 free": "free, charger-specific defaults for unclaimed sessions (to 31 Aug 2018)",
               "R2 free, 14 kWh default": "free, 14 kWh default for unclaimed sessions (1 Sep - 31 Oct 2018)",
               "R3 paid": "paid $0.12/kWh, unclaimed sessions stopped after 30 min (1 Nov 2018 - COVID start)",
               "R4 COVID period": "campus in remote operation and the recovery afterwards (from COVID start)"}
def regime_of(ts):
    ts = pd.DatetimeIndex(ts) if not isinstance(ts, pd.Series) else ts
    return np.select([ts < R2_START, ts < PAID_START, ts < COVID_START], REGIMES[:3], default=REGIMES[3])
df["regime"] = pd.Categorical(regime_of(df["connectionTime_loc"]), categories=REGIMES, ordered=True)

# Long sessions (> 72 h) removed in Issue #9 still occupied a charger: add them back for occupancy only
FLAGGED_FILE = ROOT / "acn_flagged_sessions.csv"
long_sessions = pd.DataFrame(columns=["connectionTime_loc", "disconnectTime_loc"])
if os.path.exists(FLAGGED_FILE):
    _fl = pd.read_csv(FLAGGED_FILE)
    _fl = _fl[_fl["drop_reason"].astype(str).str.startswith("connected >")]
    for c in ["connectionTime", "disconnectTime"]:
        _fl[c + "_loc"] = pd.to_datetime(_fl[c], utc=True).dt.tz_convert(LOCAL_TZ).dt.tz_localize(None)
    long_sessions = _fl[["connectionTime_loc", "disconnectTime_loc"]]
print(f"Stay split at {SHORT_STAY_H:g} h (valley ~{valley_h:.2f} h) | long sessions added back for occupancy: {len(long_sessions)}")

all_days = pd.date_range(df["date"].min(), df["date"].max(), freq="D")
n_days = len(all_days)

# --- Gap detection: >= MIN_GAP_DAYS consecutive days with no sessions = outage / missing data ---
MIN_GAP_DAYS = 7
daily_counts_all = df.groupby("date").size().reindex(all_days, fill_value=0)
is_zero = daily_counts_all == 0
run_id = (is_zero != is_zero.shift()).cumsum()
run_len = is_zero.groupby(run_id).transform("sum")
gap_mask = is_zero & (run_len >= MIN_GAP_DAYS)
gap_dates = set(all_days[gap_mask.values])
gap_runs = (pd.DataFrame({"date": all_days[gap_mask.values], "run": run_id[gap_mask].values})
            .groupby("run")["date"].agg(gap_start="min", gap_end="max", days="count").reset_index(drop=True))

daily_counts = daily_counts_all[~gap_mask]                       # days with data ("service days")
dow_of_days = pd.Series(daily_counts.index.dayofweek, index=daily_counts.index)
n_service_days = len(daily_counts)
print(f"{len(df):,} sessions | {df['stationID'].nunique()} stations | {all_days.min().date()} to {all_days.max().date()} ({n_days} calendar days, {n_service_days} with data)")
print(f"Gap runs (>= {MIN_GAP_DAYS} consecutive zero-session days): {len(gap_runs)} covering {int(gap_mask.sum())} days")

# Check the assumed COVID start against the data: first week (from 2020) that starts a run of at least 4 weeks whose
# weekday volume is below half of the median of the 8 weeks before the run. A single quiet week (e.g. the winter break)
# is a holiday, not a regime change, so it does not count.
PERSIST_WEEKS = 4
_wd = daily_counts[dow_of_days < 5]
_wk = _wd.resample("W-SUN").mean()
_ref = _wk.shift(1).rolling(8, min_periods=4).median()
_low = pd.Series([bool((_wk.iloc[i:i + PERSIST_WEEKS] < 0.5 * _ref.iloc[i]).all()) and i + PERSIST_WEEKS <= len(_wk)
                  for i in range(len(_wk))], index=_wk.index)
_brk = _wk[(_wk.index >= "2020-01-01") & _low]
covid_detected = _brk.index[0] - pd.Timedelta(days=6) if len(_brk) else None
print("Assumed COVID start:", COVID_START.date(), f"| first week starting {PERSIST_WEEKS}+ weeks with weekday volume < 50% of the previous 8 weeks:",
      covid_detected.date() if covid_detected is not None else "none found")

# ---- Analysis scope: full data kept as *_full / df_all; df and daily_counts below = normal operation (before COVID) ----
df_all = df.copy()
long_sessions_all = long_sessions
daily_counts_full, dow_of_days_full = daily_counts.copy(), dow_of_days.copy()
all_days_full, n_days_full, n_service_days_full = all_days, n_days, n_service_days
df = df_all[df_all["connectionTime_loc"] < COVID_START].copy()
long_sessions = long_sessions_all[long_sessions_all["connectionTime_loc"] < COVID_START]
all_days = all_days_full[all_days_full < COVID_START]; n_days = len(all_days)
daily_counts = daily_counts_full[daily_counts_full.index < COVID_START]
dow_of_days = dow_of_days_full[dow_of_days_full.index < COVID_START]
n_service_days = len(daily_counts)
print(f"Normal-operation scope: {len(df):,} sessions, {all_days.min().date()} to {all_days.max().date()} ({n_service_days} days with data). "
      f"COVID period: {len(df_all) - len(df):,} sessions on {n_service_days_full - n_service_days} days with data.")

Stay split at 6 h (valley ~5.88 h) | long sessions added back for occupancy: 37
31,387 sessions | 55 stations | 2018-04-25 to 2021-09-13 (1238 calendar days, 1132 with data)
Gap runs (>= 7 consecutive zero-session days): 1 covering 106 days
Assumed COVID start: 2020-03-16 | first week starting 4+ weeks with weekday volume < 50% of the previous 8 weeks: 2020-03-16
Normal-operation scope: 27,898 sessions, 2018-04-25 to 2020-03-15 (691 days with data). COVID period: 3,489 sessions on 441 days with data.


In [2]:
def describe(s, name=None):
    s = s.dropna()
    return pd.Series({
        "count": len(s), "mean": s.mean(), "std": s.std(), "cv": s.std() / s.mean() if s.mean() else np.nan,
        "min": s.min(), "p5": s.quantile(.05), "p25": s.quantile(.25), "median": s.median(),
        "p75": s.quantile(.75), "p95": s.quantile(.95), "max": s.max(),
        "skew": s.skew(), "kurtosis": s.kurt()}, name=name)

def gini(x):
    x = np.sort(np.asarray(x, dtype=float))
    n = len(x)
    return float((2 * np.arange(1, n + 1) - n - 1).dot(x) / (n * x.sum()))

def compare(a, b, label_a, label_b):
    """Median comparison with Mann-Whitney U and a rank-biserial effect size (-1..1)."""
    a, b = a.dropna(), b.dropna()
    out = {"n_" + label_a: len(a), "n_" + label_b: len(b),
           "median_" + label_a: a.median(), "median_" + label_b: b.median(),
           "mean_" + label_a: a.mean(), "mean_" + label_b: b.mean()}
    if HAS_SCIPY and len(a) > 5 and len(b) > 5:
        u, p = stats.mannwhitneyu(a, b, alternative="two-sided")
        out["p_value"] = p
        out["effect_size_rb"] = 2 * u / (len(a) * len(b)) - 1     # >0: first group tends to be larger
    return pd.Series(out)

def effect_label(r):
    r = abs(r)
    return "negligible" if r < 0.1 else "small" if r < 0.3 else "medium" if r < 0.5 else "large"

## 0. Statistical summary of the main variables

In [3]:
summary_stats = pd.DataFrame({
    "kWh per session": describe(df["kWhDelivered"]),
    "Hours plugged in": describe(df["connected_h"]),
    "Hours charging": describe(df.loc[df["done_ok"], "charging_h"]),
    "Idle hours": describe(df["idle_h"]),
    "Idle share of stay": describe(df["idle_share"]),
    "Avg power (kW)": describe(df["avg_power_kW"]),
    "Sessions per day (days with data)": describe(daily_counts),
}).T
tab("00_summary_statistics", summary_stats)

,count,mean,std,cv,min,p5,p25,median,p75,p95,max,skew,kurtosis
kWh per session,"27,898.00",8.88,8.05,0.91,0.50,0.88,3.43,6.70,12.66,23.01,75.53,2.29,7.79
Hours plugged in,"27,898.00",5.85,4.83,0.83,0.09,0.65,2.15,5.22,8.64,12.06,71.00,3.35,27.12
Hours charging,"27,094.00",3.03,2.79,0.92,0.00,0.50,1.23,2.11,4.01,8.44,45.46,3.02,20.91
Idle hours,"27,094.00",2.82,4.09,1.45,0.00,0.00,0.00,1.14,4.87,8.93,65.09,4.32,40.12
Idle share of stay,"27,094.00",0.35,0.33,0.94,0.00,0.00,0.00,0.29,0.68,0.88,1.00,0.30,-1.49
Avg power (kW),"27,092.00",3.47,1.87,0.54,0.02,0.62,2.00,3.13,5.27,6.58,9.94,0.35,-0.94
Sessions per day (days with data),691.00,40.37,24.25,0.60,0.00,7.00,22.00,38.00,51.00,89.50,106.00,0.56,-0.29


,count,mean,std,cv,min,p5,p25,median,p75,p95,max,skew,kurtosis
kWh per session,"27,898.00",8.88,8.05,0.91,0.50,0.88,3.43,6.70,12.66,23.01,75.53,2.29,7.79
Hours plugged in,"27,898.00",5.85,4.83,0.83,0.09,0.65,2.15,5.22,8.64,12.06,71.00,3.35,27.12
Hours charging,"27,094.00",3.03,2.79,0.92,0.00,0.50,1.23,2.11,4.01,8.44,45.46,3.02,20.91
Idle hours,"27,094.00",2.82,4.09,1.45,0.00,0.00,0.00,1.14,4.87,8.93,65.09,4.32,40.12
Idle share of stay,"27,094.00",0.35,0.33,0.94,0.00,0.00,0.00,0.29,0.68,0.88,1.00,0.30,-1.49
Avg power (kW),"27,092.00",3.47,1.87,0.54,0.02,0.62,2.00,3.13,5.27,6.58,9.94,0.35,-0.94
Sessions per day (days with data),691.00,40.37,24.25,0.60,0.00,7.00,22.00,38.00,51.00,89.50,106.00,0.56,-0.29


### 0b. Data gaps (consecutive days with no sessions)

In [4]:
tab("00b_data_gaps", gap_runs)
print(f"{int(gap_mask.sum())} of {n_days} calendar days sit inside a gap and are excluded from daily statistics.")
print("Zero-session days OUTSIDE gaps (isolated closures):", int((daily_counts == 0).sum()))

,gap_start,gap_end,days
0,2020-08-04,2020-11-17,106


106 of 691 calendar days sit inside a gap and are excluded from daily statistics.
Zero-session days OUTSIDE gaps (isolated closures): 1


> A gap means the source recorded nothing for a week or more. It could be a real shutdown or a recording/collection outage, and this data cannot tell which. Either way it is not "demand was zero", so it must not enter averages. State the gap dates in the report.

> `skew` well above 0 means a long right tail (a few very large values pull the mean above the median); use medians for "typical" values. `cv` (std/mean) compares variability across variables measured in different units.

## 1. Charging-session frequency

In [5]:
by_type = pd.DataFrame({
    "All days": describe(daily_counts),
    "Weekdays": describe(daily_counts[dow_of_days < 5]),
    "Weekends": describe(daily_counts[dow_of_days >= 5]),
}).T
by_type["zero_session_days"] = [int((daily_counts == 0).sum()), int((daily_counts[dow_of_days < 5] == 0).sum()), int((daily_counts[dow_of_days >= 5] == 0).sum())]
by_type["dispersion_index"] = [daily_counts.var() / daily_counts.mean(),
                               daily_counts[dow_of_days < 5].var() / daily_counts[dow_of_days < 5].mean(),
                               daily_counts[dow_of_days >= 5].var() / daily_counts[dow_of_days >= 5].mean()]
tab("01a_daily_session_frequency", by_type)

,count,mean,std,cv,min,p5,p25,median,p75,p95,max,skew,kurtosis,zero_session_days,dispersion_index
All days,691.00,40.37,24.25,0.60,0.00,7.00,22.00,38.00,51.00,89.50,106.00,0.56,-0.29,1,14.57
Weekdays,493.00,48.34,22.18,0.46,0.00,13.00,35.00,42.00,62.00,91.00,106.00,0.59,-0.24,1,10.18
Weekends,198.00,20.53,16.63,0.81,3.00,5.00,9.00,13.00,29.75,55.15,68.00,1.18,0.14,0,13.46


,count,mean,std,cv,min,p5,p25,median,p75,p95,max,skew,kurtosis,zero_session_days,dispersion_index
All days,691.00,40.37,24.25,0.60,0.00,7.00,22.00,38.00,51.00,89.50,106.00,0.56,-0.29,1,14.57
Weekdays,493.00,48.34,22.18,0.46,0.00,13.00,35.00,42.00,62.00,91.00,106.00,0.59,-0.24,1,10.18
Weekends,198.00,20.53,16.63,0.81,3.00,5.00,9.00,13.00,29.75,55.15,68.00,1.18,0.14,0,13.46


> `dispersion_index` = variance / mean of daily counts. Near 1 means counts vary like pure random arrivals; well above 1 means days differ systematically (weekday effects, holidays, growth).

In [6]:
# Weekly and monthly frequency, complete periods only (partial first/last periods would bias the numbers)
week_start = (df["date"] - pd.to_timedelta(df["dayofweek"], unit="D"))
weekly = df.groupby(week_start).size()
def week_clean(ws):
    return ws >= all_days.min() and ws + pd.Timedelta(days=6) <= all_days.max() and not any((ws + pd.Timedelta(days=i)) in gap_dates for i in range(7))
complete_weeks = weekly[[week_clean(ws) for ws in weekly.index]]

monthly = df.groupby("month").agg(sessions=("_id", "count"), active_stations=("stationID", "nunique"),
                                  total_kwh=("kWhDelivered", "sum"), median_kwh=("kWhDelivered", "median"),
                                  median_hours=("connected_h", "median"))
svc_per_month = pd.Series(1, index=daily_counts.index).groupby(daily_counts.index.to_period("M")).sum()
monthly["service_days"] = svc_per_month.reindex(monthly.index).fillna(0).astype(int)
monthly["days_in_month"] = monthly.index.days_in_month
monthly["sessions_per_service_day"] = monthly["sessions"] / monthly["service_days"].replace(0, np.nan)
monthly["sessions_per_station_per_service_day"] = monthly["sessions_per_service_day"] / monthly["active_stations"]
# usable for trend: month lies fully inside the data range and at least 80% of its days have data
inside = ~monthly.index.isin([monthly.index[0], monthly.index[-1]])
monthly["complete_month"] = inside & (monthly["service_days"] >= 0.8 * monthly["days_in_month"])
tab("01b_monthly_frequency", monthly)

missing_months = pd.period_range(monthly.index.min(), monthly.index.max(), freq="M").difference(monthly.index)
if len(missing_months):
    print("Months with NO sessions at all:", [str(m) for m in missing_months])
print(f"Complete weeks (no gap days): {len(complete_weeks)} | mean {complete_weeks.mean():.0f} sessions/week | std {complete_weeks.std():.0f} | min {complete_weeks.min()} | max {complete_weeks.max()}")

,sessions,active_stations,total_kwh,median_kwh,median_hours,service_days,days_in_month,sessions_per_service_day,sessions_per_station_per_service_day,complete_month
month,,,,,,,,,,
2018-04,268,43,"2,388.36",6.53,4.82,6,30,44.67,1.04,False
2018-05,1774,51,"15,681.59",7.32,5.48,31,31,57.23,1.12,True
2018-06,1843,52,"16,884.73",7.89,5.22,30,30,61.43,1.18,True
2018-07,2049,53,"18,950.34",7.47,4.53,31,31,66.10,1.25,True
2018-08,2427,53,"20,583.05",7.33,4.31,31,31,78.29,1.48,True
2018-09,2315,54,"20,615.23",7.92,4.11,30,30,77.17,1.43,True
2018-10,2420,54,"22,524.17",7.91,4.57,31,31,78.06,1.45,True
2018-11,1305,53,"11,679.71",6.38,5.91,30,30,43.50,0.82,True
2018-12,875,51,"8,620.64",6.72,5.20,31,31,28.23,0.55,True


Complete weeks (no gap days): 98 | mean 282 sessions/week | std 136 | min 40 | max 575


## 2. Energy consumption patterns

In [7]:
BLOCKS = [(0, 6, "Night 00–06"), (6, 10, "Morning 06–10"), (10, 16, "Midday 10–16"), (16, 20, "Evening 16–20"), (20, 24, "Late 20–24")]
df["time_block"] = ""
for lo, hi, name in BLOCKS:
    df.loc[(df["hour"] >= lo) & (df["hour"] < hi), "time_block"] = name

def group_energy(by):
    g = df.groupby(by).agg(sessions=("_id", "count"), total_kwh=("kWhDelivered", "sum"),
                           mean_kwh=("kWhDelivered", "mean"), median_kwh=("kWhDelivered", "median"),
                           p95_kwh=("kWhDelivered", lambda s: s.quantile(.95)))
    g["share_of_sessions"] = g["sessions"] / g["sessions"].sum()
    g["share_of_energy"] = g["total_kwh"] / g["total_kwh"].sum()
    return g

tab("02a_energy_by_day_type", group_energy("day_type"))
blk_tab = group_energy("time_block").reindex([b[2] for b in BLOCKS])
blk_tab["block_hours"] = [hi - lo for lo, hi, _ in BLOCKS]
blk_tab["sessions_per_day_per_block_hour"] = blk_tab["sessions"] / blk_tab["block_hours"] / n_service_days   # blocks differ in length
tab("02b_energy_by_time_block", blk_tab)
tab("02c_energy_by_stay_type", group_energy("stay"))

,sessions,total_kwh,mean_kwh,median_kwh,p95_kwh,share_of_sessions,share_of_energy
day_type,,,,,,,
Weekday,23833,"207,818.87",8.72,6.65,22.03,0.85,0.84
Weekend,4065,"40,007.62",9.84,7.40,30.67,0.15,0.16


,sessions,total_kwh,mean_kwh,median_kwh,p95_kwh,share_of_sessions,share_of_energy,block_hours,sessions_per_day_per_block_hour
time_block,,,,,,,,,
Night 00–06,501,"5,978.62",11.93,11.23,41.45,0.02,0.02,6,0.12
Morning 06–10,12080,"110,965.36",9.19,6.96,22.80,0.43,0.45,4,4.37
Midday 10–16,8987,"71,182.98",7.92,6.34,20.94,0.32,0.29,6,2.17
Evening 16–20,4568,"40,705.60",8.91,6.87,20.93,0.16,0.16,4,1.65
Late 20–24,1762,"18,993.93",10.78,7.95,39.46,0.06,0.08,4,0.64


,sessions,total_kwh,mean_kwh,median_kwh,p95_kwh,share_of_sessions,share_of_energy
stay,,,,,,,
Long (>=6 h),12732,"141,769.72",11.13,9.18,32.76,0.46,0.57
Short (<6 h),15166,"106,056.77",6.99,5.62,16.79,0.54,0.43


,sessions,total_kwh,mean_kwh,median_kwh,p95_kwh,share_of_sessions,share_of_energy
stay,,,,,,,
Long (>=6 h),12732,"141,769.72",11.13,9.18,32.76,0.46,0.57
Short (<6 h),15166,"106,056.77",6.99,5.62,16.79,0.54,0.43


In [8]:
# Is energy per session different on weekdays vs weekends? Effect size matters more than the p-value.
wk_kwh = compare(df.loc[~df.is_weekend, "kWhDelivered"], df.loc[df.is_weekend, "kWhDelivered"], "weekday", "weekend")
tab("02d_test_weekday_vs_weekend_kwh", wk_kwh.to_frame("value"))
if "effect_size_rb" in wk_kwh:
    print("Effect size:", round(wk_kwh["effect_size_rb"], 3), "->", effect_label(wk_kwh["effect_size_rb"]))

if HAS_SCIPY:
    groups = [g["kWhDelivered"].values for _, g in df.groupby("time_block")]
    h, p = stats.kruskal(*groups)
    eps2 = h / (len(df) - 1)                      # epsilon-squared effect size
    print(f"Kruskal-Wallis across time blocks: H={h:.1f}, p={p:.2g}, epsilon^2={eps2:.3f}")

,value
n_weekday,"23,833.00"
n_weekend,"4,065.00"
median_weekday,6.65
median_weekend,7.40
mean_weekday,8.72
mean_weekend,9.84
p_value,0.00
effect_size_rb,-0.05


Effect size: -0.046 -> negligible
Kruskal-Wallis across time blocks: H=318.4, p=1.1e-67, epsilon^2=0.011


In [9]:
# Concentration of energy: how much of the total comes from the biggest sessions?
s = df["kWhDelivered"].sort_values(ascending=False)
conc = pd.DataFrame({
    "share_of_sessions": [0.01, 0.05, 0.10, 0.25, 0.50],
})
conc["share_of_energy"] = [s.head(int(len(s) * q)).sum() / s.sum() for q in conc["share_of_sessions"]]
tab("02e_energy_concentration", conc.set_index("share_of_sessions"))

,share_of_energy
share_of_sessions,
0.01,0.06
0.05,0.20
0.10,0.30
0.25,0.54
0.50,0.81


,share_of_energy
share_of_sessions,
0.01,0.06
0.05,0.20
0.10,0.30
0.25,0.54
0.50,0.81


## 3. Charging duration

In [10]:
bands = pd.cut(df["connected_h"], [0, 1, 3, 6, 9, 12, 24, np.inf],
               labels=["<1 h", "1–3 h", "3–6 h", "6–9 h", "9–12 h", "12–24 h", ">24 h"], right=False)
dur = df.groupby(bands, observed=True).agg(sessions=("_id", "count"), median_kwh=("kWhDelivered", "median"),
                                          median_idle_share=("idle_share", "median"),
                                          median_arrival_hour=("hour", "median"))
dur["share_of_sessions"] = dur["sessions"] / dur["sessions"].sum()
tab("03a_duration_bands", dur)

,sessions,median_kwh,median_idle_share,median_arrival_hour,share_of_sessions
connected_h,,,,,
<1 h,2451,1.88,0.00,15.00,0.09
1–3 h,6861,5.78,0.00,14.00,0.25
3–6 h,5854,8.15,0.29,12.00,0.21
6–9 h,7018,8.22,0.59,9.00,0.25
9–12 h,4289,9.74,0.65,8.00,0.15
12–24 h,1209,13.52,0.69,17.00,0.04
>24 h,216,13.23,0.85,15.00,0.01


,sessions,median_kwh,median_idle_share,median_arrival_hour,share_of_sessions
connected_h,,,,,
<1 h,2451,1.88,0.00,15.00,0.09
1–3 h,6861,5.78,0.00,14.00,0.25
3–6 h,5854,8.15,0.29,12.00,0.21
6–9 h,7018,8.22,0.59,9.00,0.25
9–12 h,4289,9.74,0.65,8.00,0.15
12–24 h,1209,13.52,0.69,17.00,0.04
>24 h,216,13.23,0.85,15.00,0.01


In [11]:
ok = df[df["done_ok"]]
dur_tot = pd.Series({
    "plugged-in hours (trusted-done sessions)": ok["connected_h"].sum(),
    "charging hours": ok["charging_h"].sum(),
    "idle hours": ok["idle_h"].sum(),
})
dur_tot["idle share of plugged-in hours"] = dur_tot["idle hours"] / dur_tot["plugged-in hours (trusted-done sessions)"]
tab("03b_charging_vs_idle_hours", dur_tot.to_frame("value"))

if HAS_SCIPY:
    r1, p1 = stats.spearmanr(df["kWhDelivered"], df["connected_h"])
    r2, p2 = stats.spearmanr(ok["kWhDelivered"], ok["charging_h"])
    r3, p3 = stats.spearmanr(df["hour"], df["connected_h"])
    corr = pd.DataFrame({"spearman_rho": [r1, r2, r3], "p_value": [p1, p2, p3]},
                        index=["kWh vs hours plugged in", "kWh vs hours charging", "arrival hour vs hours plugged in"])
    tab("03c_duration_correlations", corr)

,value
plugged-in hours (trusted-done sessions),"158,421.69"
charging hours,"82,147.32"
idle hours,"76,291.79"
idle share of plugged-in hours,0.48


,spearman_rho,p_value
kWh vs hours plugged in,0.37,0.00
kWh vs hours charging,0.69,0.00
arrival hour vs hours plugged in,-0.40,0.00


> Spearman rank correlation (robust to the heavy tails here). |rho| < 0.3 is weak, 0.3–0.6 moderate, > 0.6 strong. A weak kWh-vs-time-plugged-in link means long stays are mostly not about needing more energy — relevant to idle time and scheduling.

## 4. Demand by time of day

In [12]:
hourly = df.groupby("hour").agg(sessions=("_id", "count"), total_kwh=("kWhDelivered", "sum"),
                                median_kwh=("kWhDelivered", "median"), median_hours=("connected_h", "median"))
hourly = hourly.reindex(range(24), fill_value=0)
hourly["share_of_arrivals"] = hourly["sessions"] / hourly["sessions"].sum()
hourly["share_of_energy"] = hourly["total_kwh"] / hourly["total_kwh"].sum()
hourly["weekday_arrivals_per_day"] = df[~df.is_weekend].groupby("hour").size().reindex(range(24), fill_value=0) / int((dow_of_days < 5).sum())
hourly["weekend_arrivals_per_day"] = df[df.is_weekend].groupby("hour").size().reindex(range(24), fill_value=0) / int((dow_of_days >= 5).sum())
tab("04a_hourly_demand", hourly)

,sessions,total_kwh,median_kwh,median_hours,share_of_arrivals,share_of_energy,weekday_arrivals_per_day,weekend_arrivals_per_day
hour,,,,,,,,
0,116,"1,341.10",6.25,4.84,0.00,0.01,0.13,0.25
1,66,731.80,6.31,3.68,0.00,0.00,0.08,0.14
2,33,566.46,13.21,3.87,0.00,0.00,0.03,0.08
3,25,238.52,6.32,3.01,0.00,0.00,0.03,0.05
4,145,"1,565.21",11.80,8.39,0.01,0.01,0.27,0.07
5,116,"1,535.54",6.17,2.21,0.00,0.01,0.22,0.05
6,663,"6,654.74",8.85,5.77,0.02,0.03,1.28,0.17
7,1663,"16,629.83",8.22,8.83,0.06,0.07,3.05,0.80
8,5241,"48,493.38",6.64,8.37,0.19,0.20,10.18,1.13


,sessions,total_kwh,median_kwh,median_hours,share_of_arrivals,share_of_energy,weekday_arrivals_per_day,weekend_arrivals_per_day
hour,,,,,,,,
0,116,"1,341.10",6.25,4.84,0.00,0.01,0.13,0.25
1,66,731.80,6.31,3.68,0.00,0.00,0.08,0.14
2,33,566.46,13.21,3.87,0.00,0.00,0.03,0.08
3,25,238.52,6.32,3.01,0.00,0.00,0.03,0.05
4,145,"1,565.21",11.80,8.39,0.01,0.01,0.27,0.07
5,116,"1,535.54",6.17,2.21,0.00,0.01,0.22,0.05
6,663,"6,654.74",8.85,5.77,0.02,0.03,1.28,0.17
7,1663,"16,629.83",8.22,8.83,0.06,0.07,3.05,0.80
8,5241,"48,493.38",6.64,8.37,0.19,0.20,10.18,1.13


In [13]:
# Concurrency: vehicles plugged in at the same time (15-minute resolution).
# Includes the > 72 h sessions removed in Issue #9, because they occupied a charger.
occ = pd.concat([df[["connectionTime_loc", "disconnectTime_loc"]], long_sessions], ignore_index=True)
start = occ["connectionTime_loc"].dt.floor("15min")
end = occ["disconnectTime_loc"].dt.floor("15min")
events = pd.concat([pd.Series(1, index=start), pd.Series(-1, index=end)]).groupby(level=0).sum()
grid = pd.date_range(events.index.min(), events.index.max(), freq="15min")
plugged = events.reindex(grid, fill_value=0).cumsum()

pl = pd.DataFrame({"plugged": plugged})
pl["hour"] = pl.index.hour
pl["weekend"] = pl.index.dayofweek >= 5
pl = pl[~pl.index.normalize().isin(list(gap_dates))]          # exclude gap days: a gap would pull every average toward zero
conc_h = pl.groupby(["weekend", "hour"])["plugged"].agg(["mean", "max"]).unstack(0)
conc_h.columns = [f"{'weekend' if w else 'weekday'}_{m}_plugged_in" for m, w in conc_h.columns]
tab("04b_concurrency_by_hour", conc_h)

wd_mean = pl[~pl.weekend].groupby("hour")["plugged"].mean()
peak_to_avg = wd_mean.max() / wd_mean.mean()
print(f"Weekday: peak hourly mean {wd_mean.max():.1f} vehicles at {int(wd_mean.idxmax())}:00 | average over the day {wd_mean.mean():.1f} | "
      f"peak-to-average {peak_to_avg:.2f} | largest simultaneous count ever {int(plugged.max())}")

# Capacity pressure: how often are (almost) all chargers occupied?
N_STATIONS = df["stationID"].nunique()
pl["occupancy"] = pl["plugged"] / N_STATIONS
wd = pl[~pl.weekend]
wd_days = wd.index.normalize().nunique()
cap = pd.Series({
    "stations": N_STATIONS,
    "largest simultaneous count": int(plugged.max()),
    "peak occupancy (largest count / stations)": plugged.max() / N_STATIONS,
    "weekday 15-min slots with >= 90% of stations occupied": int((wd["occupancy"] >= 0.9).sum()),
    "weekday days with at least one such slot": int(wd.loc[wd["occupancy"] >= 0.9].index.normalize().nunique()),
    "share of weekdays with at least one such slot": wd.loc[wd["occupancy"] >= 0.9].index.normalize().nunique() / wd_days if wd_days else np.nan,
    "weekday mean occupancy 10:00-15:00": wd.loc[(wd.hour >= 10) & (wd.hour < 15), "occupancy"].mean(),
})
tab("04c_capacity_pressure", cap.to_frame("value"))

,weekday_mean_plugged_in,weekend_mean_plugged_in,weekday_max_plugged_in,weekend_max_plugged_in
hour,,,,
0,2.85,3.40,12,14
1,2.77,3.34,13,13
2,2.70,3.33,13,13
3,2.61,3.30,13,13
4,2.61,3.29,13,13
5,2.78,3.27,13,13
6,3.02,3.25,13,13
7,4.64,3.54,23,17
8,11.08,4.29,45,19


Weekday: peak hourly mean 28.1 vehicles at 13:00 | average over the day 12.3 | peak-to-average 2.28 | largest simultaneous count ever 51


,value
stations,54.00
largest simultaneous count,51.00
peak occupancy (largest count / stations),0.94
weekday 15-min slots with >= 90% of stations occupied,40.00
weekday days with at least one such slot,5.00
share of weekdays with at least one such slot,0.01
weekday mean occupancy 10:00-15:00,0.51


,value
stations,54.00
largest simultaneous count,51.00
peak occupancy (largest count / stations),0.94
weekday 15-min slots with >= 90% of stations occupied,40.00
weekday days with at least one such slot,5.00
share of weekdays with at least one such slot,0.01
weekday mean occupancy 10:00-15:00,0.51


## 5. Demand by day of week

In [14]:
n_each = dow_of_days.value_counts().reindex(range(7))
dow = df.groupby("dayofweek").agg(sessions=("_id", "count"), total_kwh=("kWhDelivered", "sum"),
                                  median_kwh=("kWhDelivered", "median"), median_hours=("connected_h", "median"))
dow["days_in_data"] = n_each
dow["sessions_per_day"] = dow["sessions"] / dow["days_in_data"]
dow["kwh_per_day"] = dow["total_kwh"] / dow["days_in_data"]
dow.index = DOW
tab("05a_day_of_week", dow)

if HAS_SCIPY:
    by_dow = [daily_counts[dow_of_days == k].values for k in range(7)]
    h, p = stats.kruskal(*by_dow)
    print(f"Kruskal-Wallis on daily session counts across weekdays: H={h:.1f}, p={p:.2g}, epsilon^2={h/(n_service_days-1):.3f}")
wdc = compare(daily_counts[dow_of_days < 5], daily_counts[dow_of_days >= 5], "weekday", "weekend")
tab("05b_test_weekday_vs_weekend_daily_sessions", wdc.to_frame("value"))

,sessions,total_kwh,median_kwh,median_hours,days_in_data,sessions_per_day,kwh_per_day
Mon,4492,"43,102.22",7.22,5.92,98,45.84,439.82
Tue,4977,"42,758.91",6.71,6.22,98,50.79,436.32
Wed,4931,"41,080.65",6.38,6.21,99,49.81,414.96
Thu,4808,"39,567.88",6.62,5.98,99,48.57,399.68
Fri,4625,"41,309.20",6.59,5.91,99,46.72,417.26
Sat,2100,"20,523.18",7.40,2.31,99,21.21,207.30
Sun,1965,"19,484.44",7.39,2.32,99,19.85,196.81


Kruskal-Wallis on daily session counts across weekdays: H=196.7, p=9.7e-40, epsilon^2=0.285


,value
n_weekday,493.00
n_weekend,198.00
median_weekday,42.00
median_weekend,13.00
mean_weekday,48.34
mean_weekend,20.53
p_value,0.00
effect_size_rb,0.67


,value
n_weekday,493.00
n_weekend,198.00
median_weekday,42.00
median_weekend,13.00
mean_weekday,48.34
mean_weekend,20.53
p_value,0.00
effect_size_rb,0.67


## 6. Station / site-level usage

In [15]:
n_sites = df["siteID"].nunique() if "siteID" in df.columns else 1      # siteID/clusterID were dropped in Issue #9 as constant
print("Sites:", n_sites, "| stations:", df["stationID"].nunique(), "(only one site, so site-level comparison is not possible)")

stn = df.groupby("stationID").agg(sessions=("_id", "count"), total_kwh=("kWhDelivered", "sum"),
                                  median_kwh=("kWhDelivered", "median"), median_hours=("connected_h", "median"),
                                  first_session=("date", "min"), last_session=("date", "max"))
stn["observed_days"] = (stn["last_session"] - stn["first_session"]).dt.days + 1
stn["sessions_per_observed_day"] = stn["sessions"] / stn["observed_days"]
stn = stn.sort_values("sessions", ascending=False)
tab("06a_station_usage", stn, show=False)
display(stn.head(10)); display(stn.tail(5))

Sites: 1 | stations: 54 (only one site, so site-level comparison is not possible)


,sessions,total_kwh,median_kwh,median_hours,first_session,last_session,observed_days,sessions_per_observed_day
stationID,,,,,,,,
2-39-139-28,1485,"13,535.67",6.24,2.73,2018-04-25,2020-03-15,691,2.15
2-39-131-30,1176,"10,748.00",6.37,3.31,2018-04-26,2020-03-15,690,1.70
2-39-89-25,1015,"7,724.34",4.44,4.41,2018-04-25,2020-03-12,688,1.48
2-39-129-17,974,"8,308.47",6.34,2.88,2018-04-30,2020-03-15,686,1.42
2-39-127-19,923,"7,701.36",6.11,3.11,2018-04-25,2020-03-14,690,1.34
2-39-123-23,889,"6,683.78",5.45,4.11,2018-04-25,2020-03-13,689,1.29
2-39-138-29,864,"6,759.84",5.83,2.30,2018-04-25,2020-03-15,691,1.25
2-39-125-21,838,"6,550.55",6.03,3.05,2018-04-25,2020-03-15,691,1.21
2-39-79-380,789,"6,482.68",6.71,5.90,2018-04-25,2020-03-12,688,1.15


,sessions,total_kwh,median_kwh,median_hours,first_session,last_session,observed_days,sessions_per_observed_day
stationID,,,,,,,,
2-39-129-563,88,720.01,7.77,6.82,2018-05-03,2020-02-05,644,0.14
2-39-83-387,75,468.26,4.99,2.35,2018-06-03,2020-03-11,648,0.12
2-39-125-559,74,580.16,7.68,5.19,2018-04-25,2019-11-13,568,0.13
2-39-82-384,13,52.34,2.62,1.56,2018-07-18,2019-05-18,305,0.04
2-39-82-385,7,21.22,1.37,2.14,2018-09-03,2019-05-18,258,0.03


> `observed_days` runs from a station's first to last observed session. It is a proxy for when it was in service: a station that was installed early but rarely used looks the same as a late installation. Normalising by it (`sessions_per_observed_day`) removes most of the rollout effect but not all of it.

In [16]:
share = stn["sessions"].cumsum() / stn["sessions"].sum()
top10 = min(10, len(stn))
q1, q3 = stn["sessions_per_observed_day"].quantile([.25, .75]); iqr = q3 - q1
low_f, high_f = q1 - 1.5 * iqr, q3 + 1.5 * iqr
under = stn[stn["sessions_per_observed_day"] < low_f]; over = stn[stn["sessions_per_observed_day"] > high_f]

conc_stats = pd.Series({
    "stations": len(stn),
    "gini_sessions (0 = even, 1 = one station only)": gini(stn["sessions"]),
    "gini_sessions_per_observed_day": gini(stn["sessions_per_observed_day"]),
    f"top_{top10}_stations_share_of_sessions": share.iloc[top10 - 1],
    "even_share_for_same_number_of_stations": top10 / len(stn),
    "stations_first_seen_in_first_month": int((stn["first_session"] <= all_days.min() + pd.Timedelta(days=30)).sum()),
    "under-used stations (below Tukey fence, per observed day)": len(under),
    "over-used stations (above Tukey fence, per observed day)": len(over),
})
tab("06b_station_concentration", conc_stats.to_frame("value"))
if len(under): display(under[["sessions", "observed_days", "sessions_per_observed_day"]])
if len(over): display(over[["sessions", "observed_days", "sessions_per_observed_day"]])

,value
stations,54.00
"gini_sessions (0 = even, 1 = one station only)",0.34
gini_sessions_per_observed_day,0.34
top_10_stations_share_of_sessions,0.35
even_share_for_same_number_of_stations,0.19
stations_first_seen_in_first_month,51.00
"under-used stations (below Tukey fence, per observed day)",0.00
"over-used stations (above Tukey fence, per observed day)",1.00


,sessions,observed_days,sessions_per_observed_day
stationID,,,
2-39-139-28,1485,691,2.15


## 7. Trends and anomalies

### 7a. Trend (complete months only, normalised by active stations)

In [17]:
full = monthly[monthly["complete_month"]].copy()
full["month_index"] = range(len(full))
tab("07a_complete_months", full[["sessions", "service_days", "active_stations", "sessions_per_service_day", "sessions_per_station_per_service_day", "median_kwh", "median_hours"]])

trend = {}
if HAS_SCIPY and len(full) >= 5:
    for col in ["sessions_per_service_day", "sessions_per_station_per_service_day", "median_kwh", "median_hours"]:
        rho, p = stats.spearmanr(full["month_index"], full[col])
        trend[col] = {"spearman_rho_vs_time": rho, "p_value": p}
    tab("07b_trend_tests", pd.DataFrame(trend).T)
    print(f"Based on {len(full)} complete months across regimes {', '.join(sorted(set(regime_of(full.index.to_timestamp()))))}.")

# Trend inside the paid regime only (no rule change inside the window)
full_r3 = full[(full.index.to_timestamp() >= PAID_START) & (full.index.to_timestamp() < COVID_START)].copy()
trend_r3 = None
if HAS_SCIPY and len(full_r3) >= 6:
    rho, p = stats.spearmanr(range(len(full_r3)), full_r3["sessions_per_station_per_service_day"])
    trend_r3 = {"months": len(full_r3), "rho": rho, "p": p,
                "first": full_r3["sessions_per_station_per_service_day"].iloc[0], "last": full_r3["sessions_per_station_per_service_day"].iloc[-1],
                "first_m": full_r3.index[0], "last_m": full_r3.index[-1]}
    print(f"Within R3 (paid, {len(full_r3)} complete months): Spearman rho {rho:.2f}, p={p:.2g}")

,sessions,service_days,active_stations,sessions_per_service_day,sessions_per_station_per_service_day,median_kwh,median_hours
month,,,,,,,
2018-05,1774,31,51,57.23,1.12,7.32,5.48
2018-06,1843,30,52,61.43,1.18,7.89,5.22
2018-07,2049,31,53,66.10,1.25,7.47,4.53
2018-08,2427,31,53,78.29,1.48,7.33,4.31
2018-09,2315,30,54,77.17,1.43,7.92,4.11
2018-10,2420,31,54,78.06,1.45,7.91,4.57
2018-11,1305,30,53,43.50,0.82,6.38,5.91
2018-12,875,31,51,28.23,0.55,6.72,5.20
2019-01,989,31,50,31.90,0.64,6.94,6.30


,spearman_rho_vs_time,p_value
sessions_per_service_day,-0.85,0.00
sessions_per_station_per_service_day,-0.78,0.00
median_kwh,-0.88,0.00
median_hours,0.38,0.08


Based on 22 complete months across regimes R1 free, R2 free, 14 kWh default, R3 paid.
Within R3 (paid, 16 complete months): Spearman rho -0.53, p=0.036


> Everything here is per *day with data*, so a gap month is comparable with a full month. Compare `sessions_per_service_day` with `sessions_per_station_per_service_day`: if sessions rise but sessions per station do not, the growth is rollout, not behavior. With so few months, no trend should be called significant on the p-value alone.

### 7a-2. Last 16 weeks of normal operation: weekly weekday volume, spike share and share with `userID`

In [18]:
cut = daily_counts.index.max() + pd.Timedelta(days=1)          # end of normal operation (COVID start or end of data)
w = daily_counts[(daily_counts.index < cut) & (daily_counts.index >= cut - pd.Timedelta(weeks=16)) & (daily_counts.index.dayofweek < 5)]
wk_recent = w.resample("W").mean().to_frame("weekday_sessions_per_day")
_ix = df.set_index("connectionTime_loc")
wk_recent["spike_share"] = _ix["kWhDelivered"].between(13, 14, inclusive="left").resample("W").mean().reindex(wk_recent.index)
wk_recent["share_with_userID"] = _ix["has_user_id"].astype(bool).resample("W").mean().reindex(wk_recent.index)
wk_recent["policy_change_this_week"] = [any((d - pd.Timedelta(days=6) <= p <= d) for p in POLICY_DATES) for d in wk_recent.index]
tab("07j_last_weeks", wk_recent)

,weekday_sessions_per_day,spike_share,share_with_userID,policy_change_this_week
2019-12-01,18.80,0.04,0.83,False
2019-12-08,37.40,0.02,0.84,False
2019-12-15,37.20,0.04,0.83,False
2019-12-22,33.20,0.01,0.90,False
2019-12-29,5.80,0.03,0.85,False
2020-01-05,10.60,0.01,0.79,False
2020-01-12,31.40,0.03,0.80,False
2020-01-19,43.00,0.03,0.82,False
2020-01-26,34.00,0.04,0.77,False
2020-02-02,37.00,0.02,0.82,False


,weekday_sessions_per_day,spike_share,share_with_userID,policy_change_this_week
2019-12-01,18.80,0.04,0.83,False
2019-12-08,37.40,0.02,0.84,False
2019-12-15,37.20,0.04,0.83,False
2019-12-22,33.20,0.01,0.90,False
2019-12-29,5.80,0.03,0.85,False
2020-01-05,10.60,0.01,0.79,False
2020-01-12,31.40,0.03,0.80,False
2020-01-19,43.00,0.03,0.82,False
2020-01-26,34.00,0.04,0.77,False
2020-02-02,37.00,0.02,0.82,False


> A **sudden step** in weekday volume, spike share or `userID` share in a specific week points to an external change (pricing, policy, a software or recording change); weeks containing a documented policy date are marked. A **gradual slide** points to behaviour or to the recording system. Say what the table shows; Section 7e tests the documented changes directly.

### 7b. Anomalous days (relative to the same weekday in nearby weeks)

> Days in the two weeks after a regime change can be flagged because the ±2-week baseline still contains the previous regime; check `near_regime_change` before calling them anomalies.

In [19]:
# Baseline = median of the same weekday over +/-2 weeks. This accounts for weekday effects and slow growth.
base = pd.Series(index=daily_counts.index, dtype=float)
for k in range(7):
    s = daily_counts[dow_of_days == k]
    base.loc[s.index] = s.rolling(5, center=True, min_periods=3).median()
dev = (daily_counts - base) / base.replace(0, np.nan)
mad = (dev - dev.median()).abs().median()
robust_z = 0.6745 * (dev - dev.median()) / mad if mad else dev * np.nan

try:
    from pandas.tseries.holiday import USFederalHolidayCalendar
    hol = USFederalHolidayCalendar().holidays(start=all_days.min(), end=all_days.max())
except Exception:
    hol = pd.DatetimeIndex([])

anom = pd.DataFrame({"sessions": daily_counts, "baseline": base, "pct_vs_baseline": dev * 100, "robust_z": robust_z})
anom["weekday"] = [DOW[d] for d in anom.index.dayofweek]
anom["us_federal_holiday"] = anom.index.isin(hol)
anom["near_regime_change"] = [any(pd.Timedelta(0) <= d - p <= pd.Timedelta(days=14) for p in POLICY_DATES) for d in anom.index]
flag = anom[anom["robust_z"].abs() > 3.5].sort_values("robust_z", key=abs, ascending=False)
tab("07c_anomalous_days", flag)
print(f"{len(flag)} anomalous days of {len(anom)} | of these, on a US federal holiday: {int(flag['us_federal_holiday'].sum())}")

,sessions,baseline,pct_vs_baseline,robust_z,weekday,us_federal_holiday,near_regime_change
2020-01-04,12,5.00,140.00,12.28,Sat,False,False
2020-01-06,26,11.00,136.36,11.96,Mon,False,False
2019-01-07,38,19.00,100.00,8.77,Mon,False,False
2020-02-09,12,6.00,100.00,8.77,Sun,False,False
2019-12-07,8,4.00,100.00,8.77,Sat,False,False
...,...,...,...,...,...,...,...
2019-03-02,13,22.00,-40.91,-3.59,Sat,False,False
2019-02-18,19,32.00,-40.62,-3.56,Mon,True,False
2019-10-31,22,37.00,-40.54,-3.55,Thu,False,False
2019-05-19,14,10.00,40.00,3.51,Sun,False,False


63 anomalous days of 691 | of these, on a US federal holiday: 16


### 7c. Anomalies in session values

In [20]:
e = df["kWhDelivered"]
q1, q3 = e.quantile([.25, .75]); fence = q3 + 1.5 * (q3 - q1)
anom_vals = pd.Series({
    "kWh upper Tukey fence": fence,
    "sessions above fence": int((e > fence).sum()),
    "share above fence": float((e > fence).mean()),
    "sessions plugged in > 24 h": int((df["connected_h"] > 24).sum()),
    "sessions plugged in < 15 min": int((df["connected_h"] < 0.25).sum()),
    "arrivals between 00:00 and 05:00": int((df["hour"] < 5).sum()),
    "sessions with unreliable done-time (flagged earlier)": int((~df["done_ok"]).sum()),
})
tab("07d_value_anomalies", anom_vals.to_frame("value"))

,value
kWh upper Tukey fence,26.51
sessions above fence,"1,060.00"
share above fence,0.04
sessions plugged in > 24 h,216.00
sessions plugged in < 15 min,335.00
arrivals between 00:00 and 05:00,385.00
sessions with unreliable done-time (flagged earlier),804.00


,value
kWh upper Tukey fence,26.51
sessions above fence,"1,060.00"
share above fence,0.04
sessions plugged in > 24 h,216.00
sessions plugged in < 15 min,335.00
arrivals between 00:00 and 05:00,385.00
sessions with unreliable done-time (flagged earlier),804.00


In [21]:
# The 13-14 kWh spike: compare its 1-kWh bin with the neighbouring bins
edges = np.arange(0, np.ceil(e.max()) + 1, 1)
hist = pd.cut(e, edges, right=False).value_counts().sort_index()
hist.index = [f"{int(i.left)}–{int(i.right)}" for i in hist.index]
tab("07e_kwh_histogram_1kwh_bins", hist.to_frame("sessions"), show=False)
display(hist.loc["9–10":"18–19"].to_frame("sessions") if "18–19" in hist.index else hist.head(20).to_frame("sessions"))

spike_n = int(e.between(13, 14, inclusive="left").sum())
neigh = [int(e.between(a, a + 1, inclusive="left").sum()) for a in (11, 12, 14, 15)]
spike_ratio = spike_n / np.mean(neigh) if np.mean(neigh) else np.nan
print(f"13–14 kWh bin: {spike_n} sessions vs {np.mean(neigh):.0f} average in the four neighbouring 1-kWh bins (ratio {spike_ratio:.1f}x)")

sp = df[e.between(13, 14, inclusive="left")]
rest = df[~e.between(13, 14, inclusive="left")]
spike_profile = pd.DataFrame({
    "spike sessions": [len(sp), sp["has_user_id"].mean(), sp["connected_h"].median(), sp["hour"].median(), sp["is_weekend"].mean()],
    "other sessions": [len(rest), rest["has_user_id"].mean(), rest["connected_h"].median(), rest["hour"].median(), rest["is_weekend"].mean()],
}, index=["sessions", "share with userID", "median hours plugged in", "median arrival hour", "share on weekends"])
tab("07f_spike_profile", spike_profile)

,sessions
9–10,1037
10–11,1053
11–12,1262
12–13,1010
13–14,2480
14–15,637
15–16,579
16–17,490
17–18,251
18–19,244


13–14 kWh bin: 2480 sessions vs 872 average in the four neighbouring 1-kWh bins (ratio 2.8x)


,spike sessions,other sessions
sessions,"2,480.00","25,418.00"
share with userID,0.21,0.50
median hours plugged in,6.84,5.05
median arrival hour,10.00,10.00
share on weekends,0.23,0.14


,spike sessions,other sessions
sessions,"2,480.00","25,418.00"
share with userID,0.21,0.50
median hours plugged in,6.84,5.05
median arrival hour,10.00,10.00
share on weekends,0.23,0.14


In [22]:
# Does the spike look like a vehicle type, a station effect, or a system artifact? Check when, where and at what power it occurs.
df["in_spike"] = e.between(13, 14, inclusive="left")
spike_month = df.groupby("month")["in_spike"].agg(spike_sessions="sum", share_of_month_sessions="mean")
tab("07g_spike_by_month", spike_month)

st_sp = df.groupby("stationID")["in_spike"].agg(spike_sessions="sum", sessions="count", spike_share="mean")
st_sp = st_sp[st_sp["sessions"] >= 50].sort_values("spike_share", ascending=False)
tab("07h_spike_share_by_station", st_sp.head(10), show=True)
print(f"Spike share across stations with >=50 sessions: min {st_sp['spike_share'].min():.1%}, median {st_sp['spike_share'].median():.1%}, max {st_sp['spike_share'].max():.1%}")

tab("07i_spike_avg_power", df.loc[df["in_spike"], "avg_power_kW"].describe().to_frame("avg_power_kW (spike sessions)"))

,spike_sessions,share_of_month_sessions
month,,
2018-04,27,0.10
2018-05,171,0.10
2018-06,153,0.08
2018-07,227,0.11
2018-08,512,0.21
2018-09,492,0.21
2018-10,407,0.17
2018-11,56,0.04
2018-12,45,0.05


,spike_sessions,sessions,spike_share
stationID,,,
2-39-138-566,38,175,0.22
2-39-130-564,28,138,0.20
2-39-123-557,49,251,0.20
2-39-129-563,16,88,0.18
2-39-127-561,16,89,0.18
2-39-139-567,18,107,0.17
2-39-131-565,17,104,0.16
2-39-125-559,12,74,0.16
2-39-128-18,81,568,0.14


Spike share across stations with >=50 sessions: min 3.6%, median 9.3%, max 21.7%


,avg_power_kW (spike sessions)
count,"2,453.00"
mean,3.91
std,1.87
min,0.44
25%,2.20
50%,3.48
75%,5.92
max,8.61


,avg_power_kW (spike sessions)
count,"2,453.00"
mean,3.91
std,1.87
min,0.44
25%,2.20
50%,3.48
75%,5.92
max,8.61


> How to read it: a spike share that is similar every month and at every station, at a consistent power, points to a *vehicle or charger behaviour* (many cars cap at the same energy). A share that jumps at a particular month points to a *system or policy change*. A share concentrated at a few stations points to *location*. Report what the tables show; do not assert a cause they cannot support.


### 7e. Regimes: free vs. paid charging, the default energy allowance, and the COVID period

The ACN-Data authors document two changes inside this window (Lee, Li & Low 2019, Fig. 2, Sec. 3.1.2):
- **until 31 Aug 2018** sessions without app input ("unclaimed") received a default of 42, 21 or 14 kWh over 12 h depending on the charger;
- **from 1 Sep 2018** the unclaimed default was 14 kWh over 12 h;
- **from 1 Nov 2018** claimed sessions paid $0.12/kWh and unclaimed sessions were stopped after 30 minutes.

If the 13–14 kWh spike is the 14 kWh default, it should be concentrated in sessions **without** a `userID`, rise after 1 Sep, and largely disappear after 1 Nov. Paid charging should show as a drop in daily volume and a jump in the share of claimed sessions. The tables below test these expectations; they do not assume them. They cover all four regimes (R4 = COVID period); the free-vs-paid test uses only paid days before COVID, so it measures pricing and not the closure.


In [23]:

D = df_all                                                  # all regimes; gap days are already excluded from daily_counts_full
is_sp_all = D["kWhDelivered"].between(13, 14, inclusive="left")
days_by_regime = pd.Series(1, index=daily_counts_full.index).groupby(regime_of(daily_counts_full.index)).sum()
_wdi = daily_counts_full.index[daily_counts_full.index.dayofweek < 5]
wd_days_by_regime = pd.Series(1, index=_wdi).groupby(regime_of(_wdi)).sum()

reg = D.groupby("regime", observed=False).agg(sessions=("_id", "count"), median_kwh=("kWhDelivered", "median"),
                                               median_hours=("connected_h", "median"),
                                               share_with_userID=("has_user_id", "mean"))
reg["days_with_data"] = days_by_regime.reindex(REGIMES).values
reg["weekday_sessions_per_day"] = (D[~D.is_weekend].groupby("regime", observed=False).size() / wd_days_by_regime.reindex(REGIMES)).values
reg["spike_share_all"] = D.assign(s=is_sp_all).groupby("regime", observed=False)["s"].mean()
reg["spike_share_unclaimed"] = D[~D["has_user_id"].astype(bool)].assign(s=is_sp_all).groupby("regime", observed=False)["s"].mean()
reg["spike_share_claimed"] = D[D["has_user_id"].astype(bool)].assign(s=is_sp_all).groupby("regime", observed=False)["s"].mean()
reg["median_kwh_unclaimed"] = D[~D["has_user_id"].astype(bool)].groupby("regime", observed=False)["kWhDelivered"].median()
reg["weekday_share_arrivals_17_20h"] = D[~D.is_weekend].assign(ev=lambda d: d["hour"].between(17, 19)).groupby("regime", observed=False)["ev"].mean()
is_sp = df["kWhDelivered"].between(13, 14, inclusive="left")       # normal-operation version, used in the next cell
tab("07k_policy_regimes", reg)

wd_counts = daily_counts_full[dow_of_days_full < 5]
pre = wd_counts[(wd_counts.index >= R2_START) & (wd_counts.index < PAID_START)]
post = wd_counts[(wd_counts.index >= PAID_START) & (wd_counts.index < COVID_START)]    # paid, before COVID
if len(post) > 5:
    paid_test = compare(post, pre, "paid", "free_R2")
    tab("07l_test_weekday_volume_paid_vs_free", paid_test.to_frame("value"))
    print(f"Weekday sessions/day: free (R2) median {pre.median():.0f} vs paid (R3) median {post.median():.0f} "
          f"({post.median()/pre.median()-1:+.0%}); effect {effect_label(paid_test.get('effect_size_rb', np.nan))}")
else:
    paid_test = pd.Series(dtype=float)
    print("Not enough weekdays between 1 Nov 2018 and the COVID start to compare free and paid charging.")


,sessions,median_kwh,median_hours,share_with_userID,days_with_data,weekday_sessions_per_day,spike_share_all,spike_share_unclaimed,spike_share_claimed,median_kwh_unclaimed,weekday_share_arrivals_17_20h
regime,,,,,,,,,,,
R1 free,8361,7.45,4.78,0.01,129,73.97,0.13,0.13,0.02,7.39,0.16
"R2 free, 14 kWh default",4735,7.92,4.34,0.14,61,88.51,0.19,0.21,0.06,7.23,0.19
R3 paid,14802,6.11,5.78,0.84,501,36.83,0.03,0.00,0.04,0.89,0.08
R4 COVID period,3489,7.05,1.56,0.92,441,8.74,0.03,0.00,0.03,0.88,0.15


,value
n_paid,357.00
n_free_R2,43.00
median_paid,38.00
median_free_R2,90.00
mean_paid,36.83
mean_free_R2,88.51
p_value,0.00
effect_size_rb,-0.97


Weekday sessions/day: free (R2) median 90 vs paid (R3) median 38 (-58%); effect large


In [24]:

# Before 1 Sep the default depended on the charger (42, 21 or 14 kWh). Unclaimed sessions in R1 should then
# pile up near those values, and each charger's spike share should differ much more in R1 than in R2.
r1_un = df[(df["regime"] == REGIMES[0]) & ~df["has_user_id"].astype(bool)]["kWhDelivered"]
def bin_vs_neighbours(s, lo):
    n = int(s.between(lo, lo + 1, inclusive="left").sum())
    nb_ = np.mean([s.between(a, a + 1, inclusive="left").sum() for a in (lo - 2, lo - 1, lo + 1, lo + 2)])
    return n, nb_, n / nb_ if nb_ else np.nan
peaks = pd.DataFrame([dict(zip(["bin", "sessions", "neighbour_avg", "ratio"], (f"{lo}-{lo+1} kWh", *bin_vs_neighbours(r1_un, lo))))
                      for lo in (13, 20, 41)]).set_index("bin")
tab("07m_r1_unclaimed_default_bins", peaks)

st_reg = (df[df["regime"].isin(REGIMES[:2])].assign(s=is_sp)
            .groupby(["stationID", "regime"], observed=True)["s"].agg(["mean", "size"]).reset_index())
st_reg = st_reg[st_reg["size"] >= 30]
spread = st_reg.groupby("regime", observed=True)["mean"].agg(stations="size", min="min", median="median", max="max", std="std")
tab("07n_station_spike_share_spread_by_regime", spread)


,sessions,neighbour_avg,ratio
bin,,,
13-14 kWh,1088,266.00,4.09
20-21 kWh,33,41.00,0.80
41-42 kWh,20,8.75,2.29


,stations,min,median,max,std
regime,,,,,
R1 free,51,0.04,0.13,0.25,0.06
"R2 free, 14 kWh default",51,0.05,0.19,0.36,0.07


,stations,min,median,max,std
regime,,,,,
R1 free,51,0.04,0.13,0.25,0.06
"R2 free, 14 kWh default",51,0.05,0.19,0.36,0.07


> Read 7k–7n together. The documented policies are a candidate explanation; call the spike *consistent with* the default allowance only if the tables show it (concentrated in unclaimed sessions, present after 1 Sep, much smaller after 1 Nov, and more station-specific before 1 Sep). Anything modelled on the pooled data (Issue #11 onward) should either use one regime or include the regime as a feature.

### 7f. COVID period and recovery

Monthly volume per day with data, compared with the paid regime before COVID (R3). This shows how deep the drop was, how long it lasted, and how far demand had recovered by the end of the data. Gap months have no service days and are left out.

In [25]:
mon = (pd.DataFrame({"sessions": daily_counts_full})
         .assign(month=lambda d: d.index.to_period("M")).groupby("month")["sessions"].agg(service_days="size", sessions="sum"))
mon["sessions_per_service_day"] = mon["sessions"] / mon["service_days"]
mon["regime"] = regime_of(mon.index.to_timestamp())
r3_level = daily_counts_full[(daily_counts_full.index >= PAID_START) & (daily_counts_full.index < COVID_START)].mean()
mon["pct_of_R3_level"] = mon["sessions_per_service_day"] / r3_level
covid_mon = mon[mon.index.to_timestamp() >= COVID_START.to_period("M").to_timestamp()]
tab("07o_monthly_volume_vs_paid_level", mon)
covid = {}
if len(covid_mon):
    usable = covid_mon[covid_mon["service_days"] >= 10]
    trough = usable["pct_of_R3_level"].idxmin() if len(usable) else covid_mon["pct_of_R3_level"].idxmin()
    last_full = usable.index[-2] if len(usable) >= 2 else usable.index[-1] if len(usable) else covid_mon.index[-1]   # last month may be partial
    covid = {"r3_level": r3_level, "trough_month": trough, "trough_pct": mon.loc[trough, "pct_of_R3_level"],
             "last_month": last_full, "last_pct": mon.loc[last_full, "pct_of_R3_level"],
             "days": int((regime_of(daily_counts_full.index) == REGIMES[3]).sum()),
             "below_25pct_months": int((usable["pct_of_R3_level"] < 0.25).sum())}
    print(f"R3 (paid, pre-COVID) level: {r3_level:.1f} sessions per day with data")
    print(f"Lowest COVID month: {trough} at {covid['trough_pct']:.0%} of that level; "
          f"last complete month {last_full} at {covid['last_pct']:.0%}; months below 25%: {covid['below_25pct_months']}")
else:
    print("No data after the COVID start date.")

,service_days,sessions,sessions_per_service_day,regime,pct_of_R3_level
month,,,,,
2018-04,6,268,44.67,R1 free,1.51
2018-05,31,1774,57.23,R1 free,1.94
2018-06,30,1843,61.43,R1 free,2.08
2018-07,31,2049,66.10,R1 free,2.24
2018-08,31,2427,78.29,R1 free,2.65
2018-09,30,2315,77.17,"R2 free, 14 kWh default",2.61
2018-10,31,2420,78.06,"R2 free, 14 kWh default",2.64
2018-11,30,1305,43.50,R3 paid,1.47
2018-12,31,875,28.23,R3 paid,0.96


R3 (paid, pre-COVID) level: 29.5 sessions per day with data
Lowest COVID month: 2020-05 at 2% of that level; last complete month 2021-08 at 65%; months below 25%: 8


## 8. Key findings
The cell below computes the headline numbers from the tables above and writes `eda_key_findings.md`.

In [26]:
wd_row, we_row = by_type.loc["Weekdays"], by_type.loc["Weekends"]
blk = blk_tab
busiest_block = blk["sessions_per_day_per_block_hour"].idxmax()     # per hour, because blocks differ in length
long_share = (df["connected_h"] >= SHORT_STAY_H).mean()
best_dow = dow["sessions_per_day"].idxmax(); worst_dow = dow["sessions_per_day"].idxmin()

findings = [
    f"**Scope.** {len(df_all):,} sessions at {df_all['stationID'].nunique()} stations, {all_days_full.min().date()} to {all_days_full.max().date()} ({n_days_full} calendar days, {n_service_days_full} with data), one site (open to the public; mostly campus users, so a hybrid of workplace and public charging). Findings below describe **normal operation** ({all_days.min().date()} to {all_days.max().date()}, {len(df):,} sessions on {n_service_days} days) unless they name another regime; the COVID period from {COVID_START.date()} is reported separately"
    + (f" (data-detected break: week of {covid_detected.date()})" if covid_detected is not None else "") + ". "
    + (f"The paid regime has {len(full_r3)} complete months before COVID, a full year, but a single year cannot separate seasonality from trend." if 'full_r3' in dir() and len(full_r3) >= 12
       else "Less than a full year of a single regime, so annual seasonality cannot be assessed."),
    "**Data gaps.** " + ("; ".join(f"{r.gap_start.date()} to {r.gap_end.date()} ({int(r.days)} days with no sessions)" for r in gap_runs.itertuples()) if len(gap_runs) else "none of 7+ consecutive zero-session days") + ". Gap days are excluded from daily statistics, per-day rates, the plugged-in profile and trend tests.",
    f"**Frequency.** {daily_counts.mean():.1f} sessions per day on average (median {daily_counts.median():.0f}); weekdays {wd_row['mean']:.1f} vs weekends {we_row['mean']:.1f} per day. Dispersion index (1 = random arrivals): {by_type.loc['Weekdays','dispersion_index']:.1f} on weekdays, {by_type.loc['Weekends','dispersion_index']:.1f} on weekends (the all-days value, {by_type.loc['All days','dispersion_index']:.1f}, is inflated by mixing weekdays and weekends).",
    f"**Day of week.** Busiest {best_dow} ({dow.loc[best_dow,'sessions_per_day']:.1f} sessions/day), quietest {worst_dow} ({dow.loc[worst_dow,'sessions_per_day']:.1f}).",
    f"**Energy.** Median {df['kWhDelivered'].median():.1f} kWh per session (mean {df['kWhDelivered'].mean():.1f}, skew {df['kWhDelivered'].skew():.1f}). The largest 10% of sessions deliver {conc.set_index('share_of_sessions').loc[0.10,'share_of_energy']:.0%} of all energy. Weekday-vs-weekend energy per session: {effect_label(wk_kwh['effect_size_rb']) if 'effect_size_rb' in wk_kwh else 'n/a'} effect.",
    f"**Time of day.** {busiest_block} is the busiest block per hour ({blk.loc[busiest_block,'share_of_sessions']:.0%} of sessions, {blk.loc[busiest_block,'share_of_energy']:.0%} of energy, {blk.loc[busiest_block,'sessions_per_day_per_block_hour']:.1f} arrivals per hour per day on days with data). Weekday plugged-in vehicles peak at {wd_mean.max():.1f} around {int(wd_mean.idxmax())}:00 (peak-to-average {peak_to_avg:.2f}); the largest simultaneous count was {int(plugged.max())} of {N_STATIONS} chargers ({plugged.max()/N_STATIONS:.0%}); {cap['share of weekdays with at least one such slot']:.0%} of weekdays reach ≥90% occupancy at least once (> 72 h sessions included for occupancy).",
    f"**Duration.** Median {df['connected_h'].median():.1f} h plugged in; {long_share:.0%} of sessions last {SHORT_STAY_H:g} h or more (split at the density valley between the short- and long-stay modes, ≈{valley_h:.1f} h). Idle time is {dur_tot['idle share of plugged-in hours']:.0%} of plugged-in hours (sessions with a trusted done-time).",
    f"**Stations.** Usage Gini {gini(stn['sessions']):.2f}; busiest {top10} stations carry {share.iloc[top10-1]:.0%} of sessions (an even split would be {top10/len(stn):.0%}). Per observed day, {len(under)} stations are under-used and {len(over)} over-used by the Tukey rule.",
    f"**Anomalous days (normal operation).** {len(flag)} days deviate strongly from their same-weekday baseline; {int(flag['us_federal_holiday'].sum())} fall on US federal holidays and {int(flag['near_regime_change'].sum())} in the two weeks after a regime change, where the baseline still mixes regimes. The remaining {int((~flag['us_federal_holiday'] & ~flag['near_regime_change']).sum())} are unexplained by the calendar.",
    f"**13–14 kWh spike.** {spike_n:,} sessions in that 1-kWh bin, {spike_ratio:.1f}x the average of the four neighbouring bins; its monthly share ranges {spike_month['share_of_month_sessions'].min():.0%}–{spike_month['share_of_month_sessions'].max():.0%} and the per-station share {st_sp['spike_share'].min():.0%}–{st_sp['spike_share'].max():.0%}. Cause not established on this evidence alone; see the regime finding.",
]
r = reg.copy()
r.index = [str(i) for i in r.index]
R1, R2, R3, R4 = REGIMES
spike_txt = (f"spike share is {r.loc[R2, 'spike_share_unclaimed']:.0%} of unclaimed vs {r.loc[R2, 'spike_share_claimed']:.0%} of claimed sessions in R2, "
             f"and {r.loc[R3, 'spike_share_all']:.0%} of all sessions in R3 (vs {r.loc[R2, 'spike_share_all']:.0%} in R2)")
consistent = (r.loc[R2, "spike_share_unclaimed"] > 2 * r.loc[R2, "spike_share_claimed"]) and (r.loc[R3, "spike_share_all"] < 0.5 * r.loc[R2, "spike_share_all"])
wds = {k: r.loc[k, "weekday_sessions_per_day"] for k in REGIMES}
findings.append(
    f"**Regimes.** Weekday sessions per day: {wds[R1]:.1f} (R1), {wds[R2]:.1f} (R2), {wds[R3]:.1f} (R3, paid, before COVID)"
    + (f", {wds[R4]:.1f} (R4, COVID period)" if pd.notna(wds[R4]) else "")
    + f". Paid charging changed weekday volume by {wds[R3]/wds[R2]-1:+.0%} relative to R2, measured over {int(r.loc[R3, 'days_with_data'])} paid days before COVID"
    + (f" ({effect_label(paid_test['effect_size_rb'])} effect)" if 'effect_size_rb' in paid_test else "") + ". "
    + f"Share of sessions with a `userID`: " + ", ".join(f"{r.loc[k, 'share_with_userID']:.0%} ({k.split()[0]})" for k in REGIMES if pd.notna(r.loc[k, 'share_with_userID'])) + ". "
    + f"For the 13–14 kWh spike, {spike_txt}"
    + ("; this is consistent with the 14 kWh default allowance for unclaimed sessions documented by the dataset authors." if consistent
       else "; this does not clearly match the documented default allowance, so the cause stays open.")
    + (f" R3 has only {int(r.loc[R3, 'days_with_data'])} days with data, so paid-period figures are indicative." if r.loc[R3, 'days_with_data'] < 60 else ""))
if covid:
    findings.append(
        f"**COVID period (R4).** From {COVID_START.date()} volume fell to {covid['trough_pct']:.0%} of the paid pre-COVID level ({covid['r3_level']:.1f} sessions per day) "
        f"in {covid['trough_month']}; {covid['below_25pct_months']} months stayed below 25% of that level. "
        + ("; ".join(f"No sessions were recorded from {g.gap_start.date()} to {g.gap_end.date()} ({int(g.days)} days)" for g in gap_runs.itertuples() if g.gap_start >= COVID_START) + ". " if len(gap_runs) else "")
        + f"By {covid['last_month']} demand was back to {covid['last_pct']:.0%} of the paid pre-COVID level. R4 is a different regime and must not be pooled with normal operation.")

# Evidence detail for the default allowance: what the data confirms after 1 Nov, and whether the
# charger-specific 21/42 kWh defaults described for the period before 1 Sep are visible.
r21, r42 = peaks.loc["20-21 kWh", "ratio"], peaks.loc["41-42 kWh", "ratio"]
n42 = int(peaks.loc["41-42 kWh", "sessions"])
sd_r1, sd_r2 = spread["std"].iloc[0], spread["std"].iloc[1]
r1_visible = (r21 > 1.5 or r42 > 1.5) and sd_r1 > sd_r2
findings.append(
    f"**Default allowance, evidence detail.** In R3 the spike share among unclaimed sessions is {r.loc[R3, 'spike_share_unclaimed']:.0%} "
    f"and their median energy {r.loc[R3, 'median_kwh_unclaimed']:.2f} kWh, consistent with unclaimed sessions being stopped after 30 minutes. "
    f"Before 1 Sep, unclaimed sessions show a ratio of {r21:.1f}x at 20–21 kWh and {r42:.1f}x at 41–42 kWh ({n42} sessions) versus neighbouring bins, and the "
    f"station-level spike share spread is {sd_r1:.2f} (R1) vs {sd_r2:.2f} (R2), so the charger-specific 21/42 kWh defaults described for that period are "
    + ("visible in this data." if r1_visible else "not clearly visible in this data; that part of the documented policy is not confirmed here."))

if HAS_SCIPY and "sessions_per_station_per_service_day" in trend:
    t = trend["sessions_per_station_per_service_day"]
    col = "sessions_per_station_per_service_day"
    first_m, last_m = full.iloc[0], full.iloc[-1]
    peak_m = full[col].idxmax()
    if peak_m not in (full.index[0], full.index[-1]):
        shape = (f"rose from {first_m[col]:.2f} ({full.index[0]}) to a peak of {full.loc[peak_m, col]:.2f} ({peak_m}), then fell to "
                 f"{last_m[col]:.2f} ({full.index[-1]}); a straight-line trend statistic hides this rise-then-fall shape")
    else:
        shape = f"went from {first_m[col]:.2f} ({full.index[0]}) to {last_m[col]:.2f} ({full.index[-1]})"
    a, b = int(first_m['active_stations']), int(last_m['active_stations'])
    st_change = (f"Active stations went from {a} to {b}, " + ("so rollout does not explain the change." if abs(b - a) <= 0.1 * a
                 else "a large change, so part of the shift may be chargers entering or leaving service."))
    regs = sorted(set(regime_of(full.index.to_timestamp())))
    findings.append(f"**Trend (normal operation).** Sessions per active station per day with data {shape}. {st_change} "
                    f"The window spans {', '.join(regs)}, so the overall Spearman rho ({t['spearman_rho_vs_time']:.2f}) mixes rule changes with trend."
                    + (f" Within the paid regime alone ({trend_r3['months']} complete months, {trend_r3['first_m']} to {trend_r3['last_m']}) the per-station rate went from "
                       f"{trend_r3['first']:.2f} to {trend_r3['last']:.2f} (Spearman rho {trend_r3['rho']:.2f}, p={trend_r3['p']:.2f})"
                       + (": no clear trend." if trend_r3['p'] >= 0.05 else ": a statistically detectable trend; cause not established.") if trend_r3 else ""))

text = "# EDA key findings (numbers computed from the data)\n\n" + "\n\n".join(f"- {f}" for f in findings) + "\n"
with open(ROOT / "eda_key_findings.md", "w", encoding="utf-8") as fh:
    fh.write(text)
display(Markdown(text))

# EDA key findings (numbers computed from the data)

- **Scope.** 31,387 sessions at 55 stations, 2018-04-25 to 2021-09-13 (1238 calendar days, 1132 with data), one site (open to the public; mostly campus users, so a hybrid of workplace and public charging). Findings below describe **normal operation** (2018-04-25 to 2020-03-15, 27,898 sessions on 691 days) unless they name another regime; the COVID period from 2020-03-16 is reported separately (data-detected break: week of 2020-03-16). The paid regime has 16 complete months before COVID, a full year, but a single year cannot separate seasonality from trend.

- **Data gaps.** 2020-08-04 to 2020-11-17 (106 days with no sessions). Gap days are excluded from daily statistics, per-day rates, the plugged-in profile and trend tests.

- **Frequency.** 40.4 sessions per day on average (median 38); weekdays 48.3 vs weekends 20.5 per day. Dispersion index (1 = random arrivals): 10.2 on weekdays, 13.5 on weekends (the all-days value, 14.6, is inflated by mixing weekdays and weekends).

- **Day of week.** Busiest Tue (50.8 sessions/day), quietest Sun (19.8).

- **Energy.** Median 6.7 kWh per session (mean 8.9, skew 2.3). The largest 10% of sessions deliver 30% of all energy. Weekday-vs-weekend energy per session: negligible effect.

- **Time of day.** Morning 06–10 is the busiest block per hour (43% of sessions, 45% of energy, 4.4 arrivals per hour per day on days with data). Weekday plugged-in vehicles peak at 28.1 around 13:00 (peak-to-average 2.28); the largest simultaneous count was 51 of 54 chargers (94%); 1% of weekdays reach ≥90% occupancy at least once (> 72 h sessions included for occupancy).

- **Duration.** Median 5.2 h plugged in; 46% of sessions last 6 h or more (split at the density valley between the short- and long-stay modes, ≈5.9 h). Idle time is 48% of plugged-in hours (sessions with a trusted done-time).

- **Stations.** Usage Gini 0.34; busiest 10 stations carry 35% of sessions (an even split would be 19%). Per observed day, 0 stations are under-used and 1 over-used by the Tukey rule.

- **Anomalous days (normal operation).** 63 days deviate strongly from their same-weekday baseline; 16 fall on US federal holidays and 1 in the two weeks after a regime change, where the baseline still mixes regimes. The remaining 47 are unexplained by the calendar.

- **13–14 kWh spike.** 2,480 sessions in that 1-kWh bin, 2.8x the average of the four neighbouring bins; its monthly share ranges 2%–21% and the per-station share 4%–22%. Cause not established on this evidence alone; see the regime finding.

- **Regimes.** Weekday sessions per day: 74.0 (R1), 88.5 (R2), 36.8 (R3, paid, before COVID), 8.7 (R4, COVID period). Paid charging changed weekday volume by -58% relative to R2, measured over 501 paid days before COVID (large effect). Share of sessions with a `userID`: 1% (R1), 14% (R2), 84% (R3), 92% (R4). For the 13–14 kWh spike, spike share is 21% of unclaimed vs 6% of claimed sessions in R2, and 3% of all sessions in R3 (vs 19% in R2); this is consistent with the 14 kWh default allowance for unclaimed sessions documented by the dataset authors.

- **COVID period (R4).** From 2020-03-16 volume fell to 2% of the paid pre-COVID level (29.5 sessions per day) in 2020-05; 8 months stayed below 25% of that level. No sessions were recorded from 2020-08-04 to 2020-11-17 (106 days). By 2021-08 demand was back to 65% of the paid pre-COVID level. R4 is a different regime and must not be pooled with normal operation.

- **Default allowance, evidence detail.** In R3 the spike share among unclaimed sessions is 0% and their median energy 0.89 kWh, consistent with unclaimed sessions being stopped after 30 minutes. Before 1 Sep, unclaimed sessions show a ratio of 0.8x at 20–21 kWh and 2.3x at 41–42 kWh (20 sessions) versus neighbouring bins, and the station-level spike share spread is 0.06 (R1) vs 0.07 (R2), so the charger-specific 21/42 kWh defaults described for that period are not clearly visible in this data; that part of the documented policy is not confirmed here.

- **Trend (normal operation).** Sessions per active station per day with data rose from 1.12 (2018-05) to a peak of 1.48 (2018-08), then fell to 0.63 (2020-02); a straight-line trend statistic hides this rise-then-fall shape. Active stations went from 51 to 46, so rollout does not explain the change. The window spans R1 free, R2 free, 14 kWh default, R3 paid, so the overall Spearman rho (-0.78) mixes rule changes with trend. Within the paid regime alone (16 complete months, 2018-11 to 2020-02) the per-station rate went from 0.82 to 0.63 (Spearman rho -0.53, p=0.04): a statistically detectable trend; cause not established.


## Issue #10 checklist → where it is done

| Task | Section |
|---|---|
| Analyze charging-session frequency | 1 |
| Analyze energy consumption patterns | 2 |
| Analyze charging duration | 3 |
| Analyze demand by time of day | 4 |
| Analyze demand by day/week patterns | 5 |
| Analyze station/site-level usage | 6 |
| Identify important trends and anomalies | 7 (7e regimes, 7f COVID period) |

**Deliverables:** this notebook (with outputs saved), the `eda_tables/` folder (statistical summaries), and `eda_key_findings.md`.